# 03 — Automatic Evaluation

Loads the four JSON artifacts produced by the Colab training session and computes / displays the final report metrics.

**Inputs (downloaded from the Colab session, committed under `data/processed/`):**
- `pipeline_outputs/multinews_results.json` — Multi-News E2E predictions (300 clusters)
- `evaluation_outputs/stage1_test_metrics.json` — Stage 1 metrics on CNN/DM test (from `trainer.evaluate`)
- `evaluation_outputs/stage2_test_metrics.json` — Stage 2 metrics on CNN/DM test
- `evaluation_outputs/cnn_dm_test_chained.json` — Chained Stage 1 → Stage 2 predictions on CNN/DM test (single-doc consistency)

**This notebook computes:**
1. ROUGE + BERTScore on Multi-News E2E summaries (vs reference summaries)
2. ROUGE + BERTScore on CNN/DM chained predictions (vs CNN/DM headline + highlights labels)
3. Headline–summary consistency (BERTScore) on both datasets
4. A markdown summary table comparing single-doc vs multi-doc results
5. 10 qualitative samples — best by ROUGE, worst by ROUGE, possible hallucinations

Runs **locally** — no GPU needed. BERTScore downloads `roberta-large` once (~1.4 GB) into HF cache.

In [ ]:
import json
import os
import sys
from pathlib import Path

# Make src/ importable when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

from src.evaluation import evaluator
from src.evaluation.evaluator import (
    evaluate_e2e_results, evaluate_cnn_dm_chained, compile_final_table,
    save_evaluation_report, run_full_evaluation, load_config,
)

## 1. Load saved artifacts

In [ ]:
config = load_config('../configs/config.yaml')

PATHS = {
    'stage1_test':       '../data/processed/evaluation_outputs/stage1_test_metrics.json',
    'stage2_test':       '../data/processed/evaluation_outputs/stage2_test_metrics.json',
    'cnn_dm_chained':    '../data/processed/evaluation_outputs/cnn_dm_test_chained.json',
    'multinews_results': '../data/processed/pipeline_outputs/multinews_results.json',
}

for name, p in PATHS.items():
    print(f"{name:<20} {'OK ' if os.path.exists(p) else 'MISSING '}{p}")

## 2. Compute metrics + render summary table

BERTScore on first call downloads `roberta-large` (~1.4 GB). Subsequent runs hit the cache.

In [ ]:
report = run_full_evaluation(
    stage1_test_path=PATHS['stage1_test'],
    stage2_test_path=PATHS['stage2_test'],
    cnn_dm_chained_path=PATHS['cnn_dm_chained'],
    multinews_results_path=PATHS['multinews_results'],
    config=config,
    output_dir='../data/processed/evaluation_outputs',
)
print('Report keys:', list(report.keys()))

In [ ]:
from IPython.display import Markdown
Markdown(report['summary_table_md'])

## 3. Qualitative inspection — Multi-News E2E

Best/worst summaries by ROUGE-L vs the reference summary. Reads the saved Multi-News results directly.

In [ ]:
from src.evaluation.evaluator import compute_rouge

with open(PATHS['multinews_results']) as f:
    e2e = json.load(f)

# Score each cluster individually so we can rank
scored = []
for r in e2e:
    rouge = compute_rouge([r['generated_summary']], [r['reference_summary']], rouge_types=['rougeL'])
    scored.append((rouge['rougeL'], r))
scored.sort(key=lambda x: x[0])

def show(title, items):
    print(f'=== {title} ===\n')
    for s, r in items:
        print(f'-- cluster {r["cluster_id"]}  (rougeL={s:.3f}, n_articles={r["n_articles"]}) --')
        print(f'GEN HEADLINE: {r["generated_headline"]}')
        print(f'REF SUMMARY:  {r["reference_summary"][:240]}{"..." if len(r["reference_summary"]) > 240 else ""}')
        print(f'GEN SUMMARY:  {r["generated_summary"]}')
        print()

show('Top 5 — best by rougeL', list(reversed(scored[-5:])))
show('Bottom 5 — worst by rougeL', scored[:5])

## 4. Findings

- **Single-doc CNN/DM chained** vs **multi-doc Multi-News** comparison is in the summary table above. The single-doc numbers reflect a familiar in-distribution setting (`bart-large-cnn` was pretrained on CNN/DM); the multi-doc Multi-News numbers are the out-of-distribution test of the chained pipeline.
- **Headline–summary consistency** is reported on both single-doc and multi-doc — a reference-free check that the two stages agree with each other.
- **Multi-News headline-vs-reference is intentionally not scored** — Multi-News has no native headline labels, and our `reference_headline` is the first sentence of the reference summary, which would be a circular comparison target.
- The bottom-5 (worst) Multi-News cases above are good candidates for the 🔍 Error Analysis section (hallucinations, off-topic generations, distribution-shift artifacts from Stage 2 seeing Stage-1-generated headlines instead of the reference first-highlight headlines it was trained on).

### What's deferred
- **Ablations** — Section 🔬 Ablation Studies (with-vs-without headline prefix; single-doc vs multi-doc input)
- **Human evaluation** — Section 👤 Human Evaluation (50–100 samples on Likert scales)
- **Error analysis** — Section 🔍 Error Analysis (hallucination cataloguing, qualitative failure modes)